# 全部浓相链：接触与伙伴

接触对象为其他浓相链；零伙伴保留，无选中链的帧缺失。参数以保存结果为准。


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Find the project among the working directory and its parents.
ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "simulations.yaml").is_file()), None
)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the project, or set ROOT explicitly")
sys.path.insert(0, str(ROOT))
import project
import result_files as files

CONFIG = ROOT / "simulations.yaml"
SELECTED = None  # None means all registered simulations.
OUTPUT = ROOT / "02_calculate/03_contacts/02_dense_chains/02_intermolecular/output"
SIMULATIONS = project.select_simulations(CONFIG, SELECTED)
RESULTS = {s["id"]: files.completed(OUTPUT / s["id"]) for s in SIMULATIONS}
SOURCES = {key: files.read_json(path / "metadata.json") for key, path in RESULTS.items()}
COLORS = {name: plt.get_cmap("tab10")(i % 10) for i, name in enumerate(RESULTS)}
FIGURES = OUTPUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
plt.rcParams.update(
    {
        "font.size": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
    }
)


def export(fig, name):
    for key, path in RESULTS.items():
        files.completed(path)
        if files.read_json(path / "metadata.json") != SOURCES[key]:
            raise ValueError("Results changed; restart and rerun the notebook")
    for extension in ("png", "pdf", "svg"):
        fig.savefig(FIGURES / f"{name}.{extension}", dpi=300, bbox_inches="tight")
    files.write_json(
        FIGURES / f"{name}.sources.json",
        {key: files.read_json(path / "metadata.json") for key, path in RESULTS.items()},
    )


In [ ]:
THRESHOLDS = sorted(
    set.intersection(*(set(meta["parameters"]["partner_thresholds"]) for meta in SOURCES.values()))
)
if not THRESHOLDS:
    raise ValueError("No shared thresholds among selected results")

frames = pd.concat(
    [pd.read_csv(p / "frame_means.csv") for p in RESULTS.values()], ignore_index=True
)
summary = pd.concat([pd.read_csv(p / "summary.csv") for p in RESULTS.values()], ignore_index=True)
display(summary)
PRIMARY_THRESHOLD = THRESHOLDS[0]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
for name in RESULTS:
    points = frames[frames.simulation_id.eq(name) & frames.threshold.eq(PRIMARY_THRESHOLD)]
    axes[0].scatter(
        points.total_contacts, points.partner_count, s=9, alpha=0.25, color=COLORS[name], label=name
    )
    stat = summary[summary.simulation_id.eq(name)].sort_values("threshold")
    axes[1].errorbar(
        stat.threshold,
        stat.partner_count_mean_frames,
        yerr=stat.partner_count_sem_blocks,
        marker="o",
        capsize=3,
        color=COLORS[name],
        label=name,
    )
axes[0].set(
    xlabel="Residue contacts per chain",
    ylabel="Partners per chain",
    title=f"Partner threshold ≥ {PRIMARY_THRESHOLD}",
)
axes[1].set(xlabel="Minimum residue contacts defining a partner", ylabel="Partners per chain")
axes[0].legend()
export(fig, "contacts_and_partners")
plt.show()
